In [52]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader,DirectoryLoader

In [53]:
# ============================================================================
# STEP 1: DOCUMENT LOADING
# ============================================================================
# Load raw text documents from the temporary directory path

from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader
)

loader = TextLoader(
    file_path="langchain_sample.txt", 
    encoding="utf-8"
)
documents = loader.load()
print(f"Successfully loaded {len(documents)} source document(s).")

Successfully loaded 1 source document(s).


In [54]:
print(documents)

[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.\nLangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.\nRetrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.\nBM25 is a traditional sparse retrieval method that scores documents based on keyword matching. Although fast, it often struggles with synonyms and semantic similari

In [55]:
# ============================================================================
# STEP 2: TEXT SPLITTING & CHUNKING
# ============================================================================
# RecursiveCharacterTextSplitter cleanly splits large files down into manageable sizes
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_spillter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_spillter.split_documents(documents)
print(f"Created {len(chunks)} structural chunks from raw documents.")

Created 6 structural chunks from raw documents.


In [56]:
print(chunks)

[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'), Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'), Document(metadata={'source': 'langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.\nBM25 is a traditional sp

In [57]:
# ============================================================================
# STEP 3: EMBEDDINGS ENGINE INITIALIZATION
# ============================================================================
# Uses OpenAI's text-embedding-ada-002 model by default to convert text into math vectors
# (Ensure your OPENAI_API_KEY environment variable is set before running)
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings()

In [58]:
# ============================================================================
# STEP 4: VECTOR STORE SETUP (DENSE RETRIEVAL LAYER)
# ============================================================================
# Initialize and build a local directory-persisted Chroma vector store index

from langchain_community.vectorstores import Chroma

persist_dir = "./chroma_db_reranking"
print(f"Indexing chunks inside ChromaDB vector store at: {persist_dir}...")

# FIX: Changed 'documnets' to 'documents'
vectorstores = Chroma.from_documents(
    documents=chunks,              
    embedding=embedding_model,
    collection_name="rag_collection",
    persist_directory=persist_dir
)

Indexing chunks inside ChromaDB vector store at: ./chroma_db_reranking...


In [59]:
from langchain_community.retrievers import BM25Retriever

dense_retiver = vectorstores.as_retriever()

In [60]:
# ============================================================================
# STEP 5: BM25 SEARCH SETUP (SPARSE RETRIEVAL LAYER)
# ============================================================================
# Initialize keyword matcher using BM25 over the exact same chunk slices
print("Initializing Sparse BM25 Keyword Search Index...")
sparse_retriever = BM25Retriever.from_documents(chunks)
sparse_retriever.k = 3  # Retrieve the top 3 keyword-matched document layers

Initializing Sparse BM25 Keyword Search Index...


In [61]:
# ============================================================================
# STEP 6: ENSEMBLE HYBRID ENGINE COMPILATION
# ============================================================================
# Melds semantic context lookup (Chroma) and precise keywords (BM25) using RRF
# Weights give 70% importance to conceptual context, 30% to word matching
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retiver,sparse_retriever],
    weights=[0.7, 0.3]
)

print("✅ Hybrid retrieval system built successfully.")

✅ Hybrid retrieval system built successfully.


In [62]:
# ============================================================================
# STEP 7 & 8: EXECUTE INITIAL RETRIEVAL & LLM RERANKING
# ============================================================================
query = "How can i use langchain to build an application with memory and tools?"
print(f"\n🔍 Searching for: '{query}'")

# 1. Fetch initial broad list of document candidates
retrieved_docs = hybrid_retriever.invoke(query)

# 2. Format them clearly with 1-based numeric bullet points for the LLM to read
doc_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_lines)

# 3. Prompt the LLM strictly to prioritize and sort indices
rerank_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question: "{question}"

Documents:
{documents}

Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.

Output format: comma-separated document indices (e.g., 2,1,3,0,...)
""")

llm = init_chat_model("openai:gpt-3.5-turbo")
rerank_chain = rerank_prompt | llm | StrOutputParser()

# 4. Get the ordering string back from the LLM (e.g., "3,1,5,4,2")
ranking_response = rerank_chain.invoke({
    "question": query,
    "documents": formatted_docs
})
print(f"🤖 LLM Reranker Order recommendation: {ranking_response}")


🔍 Searching for: 'How can i use langchain to build an application with memory and tools?'
🤖 LLM Reranker Order recommendation: 2,1,4,3


In [63]:
# ============================================================================
# STEP 8.1: PARSE LLM STRING BACK TO PYTHON OBJECTS
# ============================================================================
reranked_docs = []
try:
    # Convert string indices down into 0-indexed integers
    rank_indices = [int(x.strip()) - 1 for x in ranking_response.split(",") if x.strip().isdigit()]
    
    for idx in rank_indices:
        if 0 <= idx < len(retrieved_docs):
            reranked_docs.append(retrieved_docs[idx])
            
    # Fallback to prevent loss of documents
    for doc in retrieved_docs:
        if doc not in reranked_docs:
            reranked_docs.append(doc)
except Exception:
    # Fail-safe backup if the LLM output was malformed
    reranked_docs = retrieved_docs

In [64]:
# ============================================================================
# STEP 8.7: DISPLAY RERANKED DOCUMENTS WITH THEIR ASSIGNED RANKS
# ============================================================================

print("\n📊 --- FINAL LLM RERANKED RESULTS ---")
# 'enumerate(..., 1)' automatically tracks the rank starting from 1 instead of 0
for rank, doc in enumerate(reranked_docs, 1):
    print(f"\n==========================================")
    print(f"🥇 RANK {rank}")
    print(f"==========================================")
    print(doc.page_content)
    print(f"------------------------------------------")


📊 --- FINAL LLM RERANKED RESULTS ---

🥇 RANK 1
LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.
------------------------------------------

🥇 RANK 2
LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.
Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.
------------------------------------------

🥇 RANK 3
FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and compressed indexes, which makes it scalable for large document stores.
Agents in LangChain are chains that use LLMs to decide which too

In [51]:
# ============================================================================
# STEP 9: FINAL ANSWER SYNTHESIS
# ============================================================================
generation_prompt = PromptTemplate.from_template("""
You are an expert AI developer assistant. Answer the user's question using ONLY the provided context below. 
If the context doesn't contain the answer, politely state that you don't know.

Context:
{context}

User Question: {question}

Final Answer:
""")

generation_chain = generation_prompt | llm | StrOutputParser()

# Isolate the top 3 highest-rated documents to feed our final response context
top_n_context = "\n\n".join([doc.page_content for doc in reranked_docs[:3]])

final_answer = generation_chain.invoke({
    "question": query,
    "context": top_n_context
})

print("\n🚀 Final Synthesized Answer:")
print(final_answer)


🚀 Final Synthesized Answer:
To build an application with memory and tools using LangChain, you can utilize the components for memory and agents provided within the framework. The memory feature in LangChain allows for context retention across multiple steps in a conversation or task, making the application more coherent and stateful. Additionally, the agents in LangChain are chains that use large language models to determine which tools to use and in what order, making them suitable for multi-step tasks. By leveraging these features, you can effectively incorporate memory and tools into your application development process.


In [65]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ============================================================================
# STEP 1 & 2: LOADING & CHUNKING
# ============================================================================
# 1. Load the single source text file directly using TextLoader
loader = TextLoader(file_path="langchain_sample.txt", encoding="utf-8")
documents = loader.load()

# 2. Split the document into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)
print(f"✅ Loaded and split into {len(chunks)} chunks.")

# ============================================================================
# STEP 3 & 4: DENSE RETRIEVAL LAYER (ChromaDB Vector Store)
# ============================================================================
embedding_model = OpenAIEmbeddings()

# Built with correct argument names and valid collection_name formatting
vectorstore = Chroma.from_documents(
    documents=chunks,              
    embedding=embedding_model,
    collection_name="rag_collection",
    persist_directory="./chroma_db_reranking"
)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# ============================================================================
# STEP 5 & 6: SPARSE & HYBRID RETRIEVAL SETUP
# ============================================================================
# Sparse keyword retriever index
sparse_retriever = BM25Retriever.from_documents(chunks)
sparse_retriever.k = 5

# Merge both retrievers seamlessly using EnsembleRetriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3]
)
print("✅ Hybrid retrieval system built successfully.")

# ============================================================================
# STEP 7 & 8: EXECUTE INITIAL RETRIEVAL & LLM RERANKING
# ============================================================================
query = "How can i use langchain to build an application with memory and tools?"
print(f"\n🔍 Searching for: '{query}'")

# 1. Fetch initial broad list of document candidates
retrieved_docs = hybrid_retriever.invoke(query)

# 2. Format them clearly with 1-based numeric bullet points for the LLM to read
doc_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_lines)

# 3. Prompt the LLM strictly to prioritize and sort indices
rerank_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question: "{question}"

Documents:
{documents}

Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.

Output format: comma-separated document indices (e.g., 2,1,3,0,...)
""")

llm = init_chat_model("openai:gpt-3.5-turbo")
rerank_chain = rerank_prompt | llm | StrOutputParser()

# 4. Get the ordering string back from the LLM (e.g., "3,1,5,4,2")
ranking_response = rerank_chain.invoke({
    "question": query,
    "documents": formatted_docs
})
print(f"🤖 LLM Reranker Order recommendation: {ranking_response}")

# ============================================================================
# STEP 8.5: PARSE LLM STRING AND ASSIGN CODES TO PARTICULAR POSITIONS
# ============================================================================
reranked_docs = []
try:
    # Convert string indices (1-indexed from prompt) down into 0-indexed integers
    rank_indices = [int(x.strip()) - 1 for x in ranking_response.split(",") if x.strip().isdigit()]
    
    # Reassign each particular document to its recommended slot
    for idx in rank_indices:
        if 0 <= idx < len(retrieved_docs):
            reranked_docs.append(retrieved_docs[idx])
            
    # Guard-rail: If the LLM left out any candidates, safely append them to the end
    for doc in retrieved_docs:
        if doc not in reranked_docs:
            reranked_docs.append(doc)
            
except Exception as e:
    print(f"⚠️ Parsing failed, falling back to original retriever order: {e}")
    reranked_docs = retrieved_docs

# ============================================================================
# STEP 8.7: DISPLAY RERANKED DOCUMENTS WITH THEIR ASSIGNED RANKS
# ============================================================================
print("\n📊 --- FINAL LLM RERANKED RESULTS WITH CONTENT ---")
for rank, doc in enumerate(reranked_docs, 1):
    print(f"\n==========================================")
    print(f"🥇 RANK {rank}")
    print(f"==========================================")
    print(doc.page_content)
    print(f"------------------------------------------")

# ============================================================================
# STEP 9: FINAL ANSWER GENERATION (THE "RAG" SYNTHESIS)
# ============================================================================
generation_prompt = PromptTemplate.from_template("""
You are an expert AI developer assistant. Answer the user's question using ONLY the provided context below. 
If the context doesn't contain the answer, politely state that you don't know.

Context:
{context}

User Question: {question}

Final Answer:
""")

generation_chain = generation_prompt | llm | StrOutputParser()

# Pass only the top 3 highest-rated documents to feed our final response context
top_n_context = "\n\n".join([doc.page_content for doc in reranked_docs[:3]])

final_answer = generation_chain.invoke({
    "question": query,
    "context": top_n_context
})

print("\n🚀 Final Synthesized Answer:")
print(final_answer)

✅ Loaded and split into 6 chunks.
✅ Hybrid retrieval system built successfully.

🔍 Searching for: 'How can i use langchain to build an application with memory and tools?'
🤖 LLM Reranker Order recommendation: 2,1,5,4,3

📊 --- FINAL LLM RERANKED RESULTS WITH CONTENT ---

🥇 RANK 1
LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.
------------------------------------------

🥇 RANK 2
LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.
Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.
------------------------------------------

🥇 RANK 3
Retrieval-Augmented Generation (RAG